# Member 3 — Classification Track, Part A (Adult / Census Income)

**Owner:** Vikranth  |  **Review 1 scope:** Classification Part A — Logistic Regression, KNN, Gaussian Naive Bayes,
Decision Tree, SVC.

- Dataset: UCI Adult (`data/raw/adult/adult_train.csv` + `adult_test.csv`, 48,842 rows, 14 features).
- Target: `income` (`<=50K` / `>50K`), binary classification.
- Pipeline: audit -> cleaning -> EDA -> outliers -> feature engineering -> encoding / stratified 80:20 split / scaling
  -> 5 models -> tuning -> evaluation (accuracy, precision, recall, weighted F1, ROC-AUC, confusion matrix) -> 5-fold CV.
- Reproducibility: `random_state=42` everywhere; every scaler/encoder is fitted on the training set only.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='colorblind')
PALETTE = {'<=50K': '#0072B2', '>50K': '#D55E00'}   # colour-blind-safe pair used for the target everywhere

RESULTS_DIR = Path('../results/classification')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / name, dpi=150, bbox_inches='tight')

## 1. Load data and audit

UCI ships the Adult data as a fixed train/test pair (32,561 + 16,281 rows). The guidelines ask for one consistent,
**stratified 80:20** split, so the two files are pooled here and re-split later (section 6). The audit below reports
shape, dtypes, missing values (`?` is this dataset's missing marker), duplicates and the class distribution.

In [2]:
raw_train = pd.read_csv('../data/raw/adult/adult_train.csv')
raw_test = pd.read_csv('../data/raw/adult/adult_test.csv')
df = pd.concat([raw_train, raw_test], ignore_index=True)

print('train file:', raw_train.shape, '| test file:', raw_test.shape, '| pooled:', df.shape)
print('duplicate rows:', df.duplicated().sum())
print('\nlabel values before cleaning:', sorted(df['income'].unique()))

audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_unique': df.nunique(),
    "'?' count": (df == '?').sum(),
    'NaN count': df.isna().sum(),
})
display(audit)
display(df.head())

train file: (32561, 15) | test file: (16281, 15) | pooled: (48842, 15)
duplicate rows: 52

label values before cleaning: ['<=50K', '>50K']


,dtype,n_unique,'?' count,NaN count
age,int64,74,0,0
workclass,object,9,2799,0
fnlwgt,int64,28523,0,0
education,object,16,0,0
education-num,int64,16,0,0
marital-status,object,7,0,0
occupation,object,15,2809,0
relationship,object,6,0,0
race,object,5,0,0
sex,object,2,0,0


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Cleaning

Strategy and justification:
- **`?` -> missing -> drop the row.** Only `workclass`, `occupation` and `native-country` contain `?`, and they are
  categorical, so a median/mean is meaningless and a "?" category would be noise. About 7% of rows are affected, so
  dropping loses little and avoids inventing values.
- **Whitespace and label normalisation.** Text columns are stripped and a trailing `.` on the label
  (`>50K.` in the original test file) is removed so both classes have exactly two spellings.
- **Duplicates removed** (identical rows carry no extra information and can leak between train and test).

In [3]:
df = df.replace('?', np.nan)

for col in df.select_dtypes(exclude='number').columns:
    df[col] = df[col].str.strip()
df['income'] = df['income'].str.rstrip('.')

n0 = len(df)
df = df.dropna().reset_index(drop=True)
n1 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n2 = len(df)

print(f'rows: {n0} -> {n1} after dropping missing ({n0 - n1} removed) -> {n2} after dropping duplicates ({n1 - n2} removed)')
print('label values after cleaning:', sorted(df['income'].unique()))
print('remaining NaN:', int(df.isna().sum().sum()))

rows: 48842 -> 45222 after dropping missing (3620 removed) -> 45175 after dropping duplicates (47 removed)
label values after cleaning: ['<=50K', '>50K']
remaining NaN: 0
